In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp04/ex_8.py ──
"""
Shared infrastructure for MLFP04 Exercise 8 — Deep Learning Foundations.

Contains: synthetic XOR data, synthetic Singapore-medical image data,
reusable training loops, gradient monitoring helpers, ModelVisualizer
output paths. Technique-specific code (model classes, per-file training
loops, scenario narratives) does NOT belong here — it lives per file.
"""

from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from kailash_ml import ModelVisualizer


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT — seeds, device, output dir
# ════════════════════════════════════════════════════════════════════════
setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

OUTPUT_DIR = Path("outputs") / "ex8_deep_learning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Shared hyperparameters
N_FEATS_XOR = 4
N_XOR_SAMPLES = 200
N_IMG_SAMPLES = 5000
IMG_SIZE = 64
N_CHANNELS = 1
N_CLASSES = 5
BATCH_SIZE = 64

# Kailash visualiser (used by every phase 4 block)
viz = ModelVisualizer()

# ════════════════════════════════════════════════════════════════════════
# DATA — XOR toy problem (Tasks 1-3)
# ════════════════════════════════════════════════════════════════════════


def make_xor_data(
    n_samples: int = N_XOR_SAMPLES, n_features: int = N_FEATS_XOR, seed: int = 42
) -> tuple[torch.Tensor, torch.Tensor, np.ndarray]:
    """Generate a synthetic XOR classification task.

    Label is XOR of the sign of features 0 and 1. Features 2..n-1 are noise.
    Returns (X_tensor, y_tensor, y_numpy) on CPU.
    """
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n_samples, n_features)).astype(np.float32)
    y = ((X[:, 0] > 0) ^ (X[:, 1] > 0)).astype(np.float32)
    X_t = torch.from_numpy(X)
    y_t = torch.from_numpy(y).unsqueeze(1)
    return X_t, y_t, y


# ════════════════════════════════════════════════════════════════════════
# DATA — Synthetic Singapore Hospital imaging tensors (Tasks 4-10)
# ════════════════════════════════════════════════════════════════════════
# Scenario: NUH (National University Hospital) chest-film triage. The real
# pipeline uses anonymised 512x512 DICOMs; this exercise uses 64x64 random
# tensors with the same multi-label structure so training completes in
# minutes on a laptop CPU / Colab T4.

SG_HOSPITAL_CLASSES = [
    "pneumonia",
    "effusion",
    "atelectasis",
    "nodule",
    "normal",
]


def make_sg_imaging_data(
    n_samples: int = N_IMG_SAMPLES, seed: int = 42
) -> tuple[np.ndarray, np.ndarray]:
    """Return (X_images, y_labels) as float32 numpy arrays.

    X: (N, 1, 64, 64) — simulated single-channel chest film tensors.
    y: (N, 5) — multi-label (~15% positive per class).
    """
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n_samples, N_CHANNELS, IMG_SIZE, IMG_SIZE)).astype(
        np.float32
    )
    y = (rng.random((n_samples, N_CLASSES)) > 0.85).astype(np.float32)
    return X, y


def build_sg_loaders(
    batch_size: int = BATCH_SIZE,
) -> tuple[DataLoader, DataLoader, np.ndarray, np.ndarray]:
    """Produce (train_loader, test_loader, X_test_np, y_test_np) for the CNN tasks."""
    X, y = make_sg_imaging_data()
    split = int(0.8 * len(X))
    X_tr, X_te = X[:split], X[split:]
    y_tr, y_te = y[:split], y[split:]

    train_ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
    test_ds = TensorDataset(torch.from_numpy(X_te), torch.from_numpy(y_te))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)
    return train_loader, test_loader, X_te, y_te


# ════════════════════════════════════════════════════════════════════════
# TRAINING UTILITIES
# ════════════════════════════════════════════════════════════════════════


def train_xor_net(
    net: nn.Module,
    X: torch.Tensor,
    y: torch.Tensor,
    optimiser: torch.optim.Optimizer,
    n_epochs: int = 100,
    criterion: nn.Module | None = None,
) -> list[float]:
    """Fit a small binary classifier to XOR data. Returns per-epoch loss."""
    crit = criterion or nn.BCEWithLogitsLoss()
    losses: list[float] = []
    for _ in range(n_epochs):
        optimiser.zero_grad()
        loss = crit(net(X), y)
        loss.backward()
        optimiser.step()
        losses.append(loss.item())
    return losses


def xor_accuracy(net: nn.Module, X: torch.Tensor, y_np: np.ndarray) -> float:
    """Binary accuracy on XOR data (threshold at 0.5)."""
    net.eval()
    with torch.no_grad():
        probs = torch.sigmoid(net(X)).numpy().flatten()
    return float(((probs > 0.5) == y_np).mean())


def grad_norm(model: nn.Module) -> float:
    """L2 norm of the concatenated gradient vector."""
    total = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total += p.grad.data.norm(2).item() ** 2
    return float(total**0.5)


def train_cnn_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimiser: torch.optim.Optimizer,
    criterion: nn.Module,
    clip_value: float | None = None,
) -> tuple[float, float]:
    """Train for one epoch on the Singapore imaging loader.

    Returns (mean_loss, mean_grad_norm). If ``clip_value`` is set, the grad
    norm is measured pre-clipping and ``clip_grad_norm_`` is applied.
    """
    model.train()
    losses: list[float] = []
    grads: list[float] = []
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimiser.zero_grad()
        loss = criterion(model(X_b), y_b)
        loss.backward()
        grads.append(grad_norm(model))
        if clip_value is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_value)
        optimiser.step()
        losses.append(loss.item())
    return float(np.mean(losses)), float(np.mean(grads))


def eval_cnn(model: nn.Module, loader: DataLoader, criterion: nn.Module) -> float:
    """Return mean validation loss across the loader."""
    model.eval()
    losses: list[float] = []
    with torch.no_grad():
        for X_b, y_b in loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            losses.append(criterion(model(X_b), y_b).item())
    return float(np.mean(losses))


# ════════════════════════════════════════════════════════════════════════
# CNN BUILDING BLOCKS (reused across files 03, 04, 05)
# ════════════════════════════════════════════════════════════════════════


class ResBlock(nn.Module):
    """Residual block: skip connection preserves gradient flow."""

    def __init__(self, channels: int) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # noqa: D401
        residual = x
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return torch.relu(out + residual)


class TriageCNN(nn.Module):
    """CNN for multi-label Singapore hospital triage.

    Architecture: Conv32 -> ResBlock -> Conv64 -> ResBlock -> AdaptiveAvgPool
    -> Dropout -> Linear. Designed for the multi-label BCEWithLogitsLoss
    setup used throughout Exercise 8.
    """

    def __init__(self, n_classes: int = N_CLASSES, dropout_rate: float = 0.3) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            ResBlock(32),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            ResBlock(64),
            nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # noqa: D401
        return self.classifier(self.features(x))


def count_params(model: nn.Module) -> tuple[int, int]:
    """Return (total_params, trainable_params)."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# ════════════════════════════════════════════════════════════════════════
# DATA LOADER ENTRY POINT
# ════════════════════════════════════════════════════════════════════════
# We expose an MLFPDataLoader handle so student files have a single import
# path even though the tensors are generated on the fly. Real datasets for
# CNN fine-tuning live in Module 5.
loader = MLFPDataLoader()




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 8.2: Activations and Weight Initialisation
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Compare ReLU, GELU, Tanh, and SiLU on the same task
#   - See why zero init and unscaled normal init fail
#   - Apply Xavier (Sigmoid/Tanh) and Kaiming (ReLU-family) correctly
#
# PREREQUISITES: 01_xor_proof.py
#
# ESTIMATED TIME: ~35 min
#
# TASKS:
#   1. Theory — paired decision: activation and init must agree
#   2. Build — network factory + init application helper
#   3. Train — activation grid and initialisation grid
#   4. Visualise — two side-by-side loss charts
#   5. Apply — Grab Singapore: Kaiming init halved their retraining budget
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import torch
import torch.nn as nn


print("\n" + "=" * 70)
print("  Activations and Initialisation — The Two Knobs That Must Agree")
print("=" * 70)



## THEORY — Activation and init are paired


In [ ]:
# ReLU kills half the signal, so the surviving half needs twice the
# variance to keep activations stable across layers. Kaiming init
# (Var(W) = 2/fan_in) compensates. Xavier (Var(W) = 2/(fan_in+fan_out))
# is tuned for tanh/sigmoid. Zeros breaks by symmetry; unscaled normal
# blows up the pre-activation.

X, y, y_np = make_xor_data()



## TASK 2 — BUILD: factory + init helper


Two-hidden-layer MLP with a swappable activation.


Apply an initialisation to every linear layer.


In [ ]:
def build_net(activation: nn.Module) -> nn.Sequential:
    # TODO: Return an nn.Sequential with layers:
    #   Linear(N_FEATS_XOR, 32) -> activation -> Linear(32, 16) -> activation -> Linear(16, 1)
    return ____


def apply_init(net: nn.Sequential, init_fn) -> None:
    # TODO: Iterate net.modules(). For each nn.Linear, call init_fn on
    # m.weight and nn.init.zeros_ on m.bias.
    ____



## TASK 3 — TRAIN the activation grid


In [ ]:
print("\n--- Activation comparison (Kaiming init, Adam, 80 epochs) ---")

activation_variants: dict[str, nn.Module] = {
    "ReLU": nn.ReLU(),
    "GELU": nn.GELU(),
    "Tanh": nn.Tanh(),
    "SiLU/Swish": nn.SiLU(),
}

act_histories: dict[str, list[float]] = {}
for name, act in activation_variants.items():
    net = build_net(act)
    apply_init(net, lambda w: nn.init.kaiming_uniform_(w, nonlinearity="relu"))
    # TODO: Train with Adam lr=0.01 for 80 epochs via train_xor_net
    losses = ____
    acc = xor_accuracy(net, X, y_np)
    act_histories[name] = losses
    print(f"  {name:<12}: final_loss={losses[-1]:.4f}, accuracy={acc:.4f}")

assert all(
    h[-1] < h[0] for h in act_histories.values()
), "Task 3: every activation should have reduced its loss"

# Initialisation grid (ReLU fixed, init swapped)
print("\n--- Initialisation comparison (ReLU, Adam, 80 epochs) ---")

init_variants = {
    "Xavier/Glorot": lambda w: nn.init.xavier_uniform_(w),
    "Kaiming/He": lambda w: nn.init.kaiming_uniform_(w, nonlinearity="relu"),
    "Normal(0,1)": lambda w: nn.init.normal_(w, mean=0.0, std=1.0),
    "Zeros": lambda w: nn.init.zeros_(w),
}

init_histories: dict[str, list[float]] = {}
for name, init_fn in init_variants.items():
    net = build_net(nn.ReLU())
    # TODO: Apply this init's fn via apply_init, then train for 80 epochs
    # with Adam lr=0.01
    apply_init(____, ____)
    losses = train_xor_net(
        net, X, y, torch.optim.Adam(net.parameters(), lr=0.01), n_epochs=80
    )
    init_histories[name] = losses
    print(f"  {name:<15}: init_loss={losses[0]:.4f}, final_loss={losses[-1]:.4f}")

assert (
    init_histories["Kaiming/He"][-1] < init_histories["Zeros"][-1] + 0.1
), "Task 3: Kaiming must beat zero init"
print("\n[ok] Checkpoint passed — activation + init grid trained\n")



## TASK 4 — VISUALISE both grids


In [ ]:
fig_act = viz.training_history(act_histories, x_label="Epoch")
fig_act.update_layout(title="Activation Comparison on XOR (Kaiming init)")
fig_act.write_html(OUTPUT_DIR / "02_activation_curves.html")

fig_init = viz.training_history(init_histories, x_label="Epoch")
fig_init.update_layout(title="Initialisation Comparison on XOR (ReLU hidden)")
fig_init.write_html(OUTPUT_DIR / "02_initialisation_curves.html")
print("[viz] 02_activation_curves.html + 02_initialisation_curves.html saved")



## TASK 5 — APPLY: Grab Rider Churn Scoring (Singapore + SEA)


In [ ]:
# SCENARIO: Grab's driver retention team scores ~200K riders nightly. A
# 2023 re-architecture swapped Tanh+Xavier for ReLU+Kaiming in the hidden
# trunk. Epochs-to-converge dropped from 14 to 6. Training time per night
# fell from 3.5h to 1.5h. Monthly T4 compute fell from ~S$9K to ~S$3.8K.
#
# BUSINESS IMPACT: S$62K/year recovered compute plus a faster nightly SLO
# (refresh by 5am instead of 8am). One-line change: kaiming_uniform_.



## REFLECTION


[x] Compared ReLU, GELU, Tanh, SiLU on identical tasks
  [x] Reproduced the Xavier vs Kaiming vs Zeros init comparison
  [x] Saw zero init fail by symmetry
  [x] Connected the paired decision to Grab's 60% compute saving

  Next: 03_cnn_residual.py — stack the layers into a CNN with skip blocks.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

